In [ ]:
import duckdb
import leafmap
import pandas as pd

In [ ]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [ ]:
%load_ext sql

In [ ]:
# Connect to DuckDB in-memory database
con = duckdb.connect()
# Connect to DuckDB file-based database
# con = duckdb.connect('filename.db')
# Connect to DuckDB in-memory database using SQL magic
%sql duckdb:///:memory:

# the %%sql commands don't require strings, whereas the duckdb.connect() requires strings, making the magic commands a lot more convenient

In [ ]:
con.install_extension("httpfs")
con.load_extension("httpfs")
con.install_extension("spatial")
con.load_extension("spatial")

## View all extensions: con.sql("FROM duckdb_extensions();")
# this will give a DuckDB output with only first 10 and last 10 rows
# to convert to DF, use con.sql("FROM duckdb_extensions();").df() at the end of the command

In [ ]:
con.sql("FROM duckdb_extensions();")

In [ ]:
#%%sql
#INSTALL httpfs;
#LOAD httpfs;
#INSTALL spatial;
#LOAD spatial;

In [ ]:
#%%sql
#FROM duckdb_extensions();

# this will automatically convert to pandas DF because of the earlier config setting

In [ ]:
# persistent tables - downloaded from the URL and store in the cache so that we don't have to keep using the URL every time

#%%sql
#CREATE OR REPLACE TABLE cities AS SELECT * FROM 'https://opengeos.org/data/duckdb/cities.csv';

con.sql("CREATE OR REPLACE TABLE cities AS SELECT * FROM 'https://opengeos.org/data/duckdb/cities.csv';")

In [ ]:
con.sql("CREATE OR REPLACE TABLE countries AS SELECT * FROM 'https://opengeos.org/data/duckdb/countries.csv';")

In [ ]:
con.sql("SELECT COUNT(*) FROM cities;")

In [ ]:
con.sql("SELECT COUNT(DISTINCT country) FROM cities;")

In [ ]:
con.sql("SELECT MIN(population) FROM cities;")

In [ ]:
con.sql("SELECT AVG(population) FROM cities;")

In [ ]:
con.sql("SELECT MAX(population) FROM cities;")

In [ ]:
con.sql("SELECT * FROM cities WHERE population > 1000000 ORDER BY population DESC LIMIT 10;")

In [ ]:
con.sql("SELECT country, COUNT(*) as city_count, AVG(population) as avg_population FROM cities GROUP BY country ORDER BY city_count DESC LIMIT 10;").df()

In [ ]:
# output results in lists and tuples

con.sql('SELECT 42').fetchall()

In [ ]:
# output results in pandas DataFrame

con.sql('SELECT 42').df()

In [ ]:
# output in numpy arrays

con.sql('SELECT 42').fetchnumpy()

In [ ]:
url = "https://opengeos.org/data/duckdb/cities.zip"
leafmap.download_file(url, unzip=True)

In [ ]:
con.read_csv('cities.csv', parallel=True)

# parallel is for larger files - uses multiple threads to read the file faster
# but for smaller files it may not make a difference and can even be slower due to overhead of managing threads

In [ ]:
# you can also do con.sql("SELECT * FROM 'cities.csv'")

In [ ]:
con.read_parquet('cities.parquet')

In [ ]:
con.read_json('cities.json')

In [ ]:
con.sql("SELECT * FROM ST_Read('cities.geojson');")

In [ ]:
con.sql("SELECT * FROM ST_Read('cities.shp');")

In [ ]:
con.sql("""
CREATE TABLE IF NOT EXISTS cities AS
FROM 'https://opengeos.org/data/duckdb/cities.parquet'
""")

In [ ]:
con.sql("COPY cities TO 'citiesW.csv' (HEADER, DELIMITER ',')")

In [ ]:
con.sql("COPY (SELECT * FROM cities WHERE country='USA') to 'cities_us.json'")

In [ ]:
con.sql("COPY (SELECT * FROM cities WHERE country='IND') TO 'cities_ind.parquet' (FORMAT PARQUET)")

In [ ]:
con.sql("COPY (SELECT * FROM cities WHERE country='JPN') TO 'cities_jpn.geojson' WITH (FORMAT GDAL, DRIVER 'GeoJSON')")

In [ ]:
con.sql("COPY (SELECT * FROM cities WHERE country='ITA') TO 'cities_ita.shp' WITH (FORMAT GDAL, DRIVER 'ESRI Shapefile')")

In [ ]:
con.sql("COPY (SELECT * FROM cities WHERE country='BRA') TO 'cities_bra.gpkg' WITH (FORMAT GDAL, DRIVER 'GPKG')")

## NYC Data

In [ ]:
url = "https://opengeos.org/data/duckdb/nyc_data.db.zip"
leafmap.download_file(url, unzip=True)

In [ ]:
con=duckdb.connect("nyc_data.db")

In [ ]:
con.install_extension("spatial")
con.load_extension("spatial")

In [ ]:
con.sql("SELECT name, ST_AsText(geom) FROM nyc_subway_stations LIMIT 10;")

In [ ]:
con.sql("SHOW TABLES;")

In [ ]:
con.sql("SHOW VARIABLES; FROM nyc_subway_stations;")

In [ ]:
con.sql("SELECT name, boroname FROM nyc_neighborhoods WHERE ST_Intersects(geom, ST_GeomFromText('POINT(583571 4506714)'))")

In [ ]:
con.sql("SELECT name, geom FROM nyc_subway_stations WHERE name = 'Broad St';")

In [ ]:
con.sql("SELECT COUNT(*) as nearby_stations FROM nyc_subway_stations WHERE ST_DWithin(geom, ST_GeomFromText('POINT(583571 4506714)'), 500);")

In [ ]:
con.sql("SELECT subways.name AS subway_name, neighborhoods.name AS neighborhood_name, neighborhoods.boroname AS borough FROM nyc_neighborhoods AS neighborhoods JOIN nyc_subway_stations AS subways ON ST_Intersects(neighborhoods.geom, subways.geom) WHERE subways.name = 'Broad St';")

In [ ]:
con.sql("DESCRIBE nyc_subway_stations;")

# null = is it allowed as null
# key is null bc no primary key
# default is the automatic fallback value for new rows
# extra is any additional rules for that column

In [ ]:
con.sql("SELECT DISTINCT COLOR from nyc_subway_stations;").df()

In [ ]:
con.sql("SELECT subways.name AS subway_name, neighborhoods.name AS neighborhood_name, neighborhoods.boroname AS borough FROM nyc_neighborhoods AS neighborhoods JOIN nyc_subway_stations AS subways ON ST_Intersects(neighborhoods.geom, subways.geom) WHERE subways.color = 'Red';")

In [ ]:
con.sql("DESCRIBE nyc_census_blocks")

In [ ]:
con.sql("""
SELECT
        100*SUM(popn_white)/SUM(popn_total) AS pct_white,
        100*SUM(popn_black)/SUM(popn_total) AS pct_black,
        100*SUM(popn_asian)/SUM(popn_total) AS pct_asian,
        100*SUM(popn_nativ)/SUM(popn_total) AS pct_native,
        SUM(popn_total) AS total_population
    FROM nyc_census_blocks;""")

In [ ]:
con.sql("""
    SELECT DISTINCT routes FROM nyc_subway_stations AS subways,
    WHERE strpos(subways.routes,'A') > 0;
    """)

In [ ]:
con.sql("""
SELECT
    100*SUM(popn_white)/SUM(popn_total) AS white_pct,
    100*SUM(popn_black)/SUM(popn_total) AS black_pct,
    100*SUM(popn_asian)/SUM(popn_total) AS asian_pct,
    100*SUM(popn_nativ)/SUM(popn_total) AS native_pct,
    SUM(popn_total) AS popn_total
FROM nyc_census_blocks AS census
JOIN nyc_subway_stations AS subways
ON ST_DWithin(census.geom, subways.geom, 200)
WHERE strpos(subways.routes,'A') > 0;
""")

In [ ]:
con.sql("""
WITH subway_lines AS (
    SELECT DISTINCT trim(unnest(string_split(routes,','))) AS route
    FROM nyc_subway_stations
    WHERE routes is NOT NULL)
SELECT
    lines.route,
    100*SUM(popn_white)/SUM(popn_total) AS white_pct,
    100*SUM(popn_black)/SUM(popn_total) AS black_pct,
    100*SUM(popn_asian)/SUM(popn_total) AS asian_pct,
    100*SUM(popn_nativ)/SUM(popn_total) AS native_pct,
    SUM(popn_total) AS popn_total
FROM nyc_census_blocks AS census
JOIN nyc_subway_stations AS subways
ON ST_DWithin(census.geom, subways.geom, 200)
JOIN subway_lines AS lines
    ON strpos(subways.routes, lines.route) > 0
GROUP BY lines.route
ORDER BY black_pct DESC;
""").df()

## National Wetland Analysis

In [ ]:
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")

In [ ]:
state = "AZ"
url = f"https://data.source.coop/giswqs/nwi/wetlands/{state}_Wetlands.parquet"
con.sql(f"SELECT * FROM '{url}'")

In [ ]:
con.sql(f"DESCRIBE FROM '{url}'")

In [ ]:
con.sql(f"""
SELECT COUNT(*) AS Count
FROM 's3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet'
""")

In [ ]:
df = con.sql(f"""
SELECT filename, COUNT(*) AS Count
FROM read_parquet('s3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet', filename=True)
GROUP BY filename
ORDER BY COUNT(*) DESC;
""").df()
df.head()

In [ ]:
count_df = con.sql(f"""
SELECT SUBSTRING(filename, LENGTH(filename) - 18, 2) AS State, COUNT(*) AS Count
FROM read_parquet('s3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet', filename=True)
GROUP BY State
ORDER BY COUNT(*) DESC;
""").df()
count_df.head(10)

In [ ]:
con.sql("CREATE OR REPLACE TABLE wetlands AS FROM count_df")
con.sql("FROM wetlands")

In [ ]:
url = 'https://opengeos.org/data/us/us_states.parquet'
con.sql(
    f"""
CREATE OR REPLACE TABLE states AS
SELECT * FROM '{url}'
"""
)

In [ ]:
con.sql("""
SELECT * FROM states INNER JOIN wetlands on states.id = wetlands.State
""")

In [ ]:
file_path = "states_with_wetlands.geojson"
con.sql("COPY (SELECT s.name, s.id, w.Count, s.geometry FROM states s JOIN wetlands w ON s.id = w.State ORDER BY w.Count DESC) TO '{file_path}' WITH (FORMAT GDAL, DRIVER 'GeoJSON')")

In [ ]:
m = leafmap.Map()
m.add_data(
    file_path, column="Count", scheme="Quantiles", cmap="Greens",
    legend_title="Wetland Count"    
)
m

In [ ]:
leafmap.pie_chart(
    count_df, "State", "Count", height=800, title="Number of Wetlands by State"
)

In [ ]:
con.sql("SELECT SUM(Shape_Area) / 1000000 AS Area_SqKm FROM 's3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet';")

In [ ]:
area_df = con.sql("SELECT SUBSTRING(filename, LENGTH(filename) - 18, 2) AS State, SUM(Shape_Area) / 1000000 AS Area_SqKm FROM read_parquet('s3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/*.parquet', filename=True) GROUP BY State ORDER BY COUNT(*) DESC;").df()
area_df.head(10)

In [ ]:
leafmap.bar_chart(area_df, "State", "Area_SqKm", title="Wetland Area by State")